# Le pivot de Gauss à la main puis à la machine

| Module | Durée | Prérequis |
|---|---|---|
| M2 – Systèmes linéaires | 75 min | M1 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une fleuriste vend trois bouquets tout faits. Elle a perdu sa fiche de prix unitaires, mais connaît la composition et le prix de chaque bouquet :

| Bouquet | Roses | Tulipes | Lys | Prix |
|---|---|---|---|---|
| Printemps | 3 | 2 | 1 | 11 € |
| Douceur | 1 | 4 | 2 | 12 € |
| Éclat | 2 | 1 | 3 | 14 € |

**Quel est le prix d'une rose, d'une tulipe et d'un lys ?**

Au notebook 2.1, `np.linalg.solve` a résolu ce type de problème comme une boîte noire. Ici, on ouvre la boîte : comment résout-on un système, et pourquoi l'outil se comporte-t-il comme il le fait ?

**Hypothèses.** La fleuriste ne fait aucune remise sur les bouquets et ne facture pas l'emballage. Est-ce plausible ? Que changerait un emballage à 1 € par bouquet ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Le bouquet Printemps contient 6 fleurs pour 11 €. Pariez sur le prix d'une rose.

In [ ]:
pari_prix_rose = 2.0   # en euros (modifiez)

## ③ Abstraire : la méthode du pivot

Avec $r$, $t$, $l$ les prix d'une rose, d'une tulipe et d'un lys :

$$
\begin{cases}
3r + 2t + l = 11 \\
r + 4t + 2l = 12 \\
2r + t + 3l = 14
\end{cases}
\qquad\Longleftrightarrow\qquad
\begin{pmatrix} 3 & 2 & 1 \\ 1 & 4 & 2 \\ 2 & 1 & 3 \end{pmatrix}
\begin{pmatrix} r \\ t \\ l \end{pmatrix}
=
\begin{pmatrix} 11 \\ 12 \\ 14 \end{pmatrix}
$$

L'idée du pivot de Gauss est d'**éliminer** les inconnues une à une, avec trois opérations qui ne changent pas les solutions :

1. échanger deux lignes ;
2. multiplier une ligne par un nombre non nul ;
3. ajouter à une ligne un multiple d'une autre ligne.

On obtient un système **triangulaire** : la dernière équation n'a plus qu'une inconnue, l'avant-dernière deux, etc. On le résout ensuite **en remontant**.

![Schéma : élimination de Gauss, de la matrice augmentée pleine à la forme triangulaire](../../assets/schemas/schema-elimination.svg)

**Activité** : Sur papier, faites l'élimination :

- utilisez la ligne 1 pour faire disparaître $r$ des lignes 2 et 3 ;
- utilisez la nouvelle ligne 2 pour faire disparaître $t$ de la ligne 3 ;
- remontez pour trouver $l$, puis $t$, puis $r$.

Conseil : les fractions sont inévitables. Gardez-les exactes, sans arrondir.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Outil 1 : le pivot pas à pas, en fractions exactes

La fonction suivante fait exactement ce que vous avez fait à la main, et affiche chaque opération. Comparez ligne par ligne avec votre feuille.

In [ ]:
from fractions import Fraction

def afficher(M):
    for ligne in M:
        *coefs, droite = ligne
        print("  [" + "  ".join(f"{str(c):>6s}" for c in coefs) + "  |" + f"{str(droite):>7s} ]")
    print()

def pivot_de_gauss(A, b, verbeux=True):
    # Échelonne la matrice augmentée [A | b] et renvoie la liste de ses lignes (en fractions exactes)
    M = [[Fraction(x) for x in ligne] + [Fraction(bi)] for ligne, bi in zip(A, b)]
    n_lignes, n_inconnues = len(M), len(M[0]) - 1
    if verbeux:
        print("Matrice augmentée de départ :"); afficher(M)
    p = 0                                                    # ligne du prochain pivot
    for col in range(n_inconnues):
        candidats = [i for i in range(p, n_lignes) if M[i][col] != 0]
        if not candidats:
            continue                                         # colonne sans pivot : on passe à la suivante
        operations = 0
        if candidats[0] != p:
            M[p], M[candidats[0]] = M[candidats[0]], M[p]
            operations += 1
            if verbeux:
                print(f"L{p+1} <-> L{candidats[0]+1}")
        for i in range(p + 1, n_lignes):
            facteur = M[i][col] / M[p][col]
            if facteur != 0:
                M[i] = [x - facteur * y for x, y in zip(M[i], M[p])]
                operations += 1
                if verbeux:
                    print(f"L{i+1} <- L{i+1} - ({facteur}) x L{p+1}")
        if verbeux and operations:
            afficher(M)
        p += 1
        if p == n_lignes:
            break
    return M

def remontee(M):
    # Résout un système triangulaire carré, de la dernière ligne vers la première
    n = len(M)
    x = [Fraction(0)] * n
    for i in reversed(range(n)):
        somme = sum(M[i][j] * x[j] for j in range(i + 1, n))
        x[i] = (M[i][n] - somme) / M[i][i]
    return x

A = [[3, 2, 1], [1, 4, 2], [2, 1, 3]]
b = [11, 12, 14]
M = pivot_de_gauss(A, b)
prix = remontee(M)
print("Prix (rose, tulipe, lys) :", [str(p) for p in prix])

**Activité** : Vos lignes intermédiaires sont-elles identiques ? Si vous avez choisi un autre ordre d'élimination, vos lignes diffèrent, mais pas la solution finale. Pourquoi ?

Indiquez vos réponses et justifiez-les.

### Outil 2 : la forme échelonnée réduite avec SymPy

SymPy, la bibliothèque de calcul exact, va plus loin : elle « réduit » aussi au-dessus des pivots, jusqu'à obtenir la matrice identité à gauche. La solution se lit alors directement dans la dernière colonne.

In [ ]:
import sympy as sp

augmentee = sp.Matrix([[3, 2, 1, 11], [1, 4, 2, 12], [2, 1, 3, 14]])
reduite, colonnes_pivots = augmentee.rref()
display(reduite)
print("Colonnes des pivots :", colonnes_pivots)

### Outil 3 : NumPy, en nombres à virgule

In [ ]:
import numpy as np

x_numpy = np.linalg.solve(np.array(A, dtype=float), np.array(b, dtype=float))
print("np.linalg.solve :", x_numpy)
print("Écart avec le calcul exact :", [float(p) - v for p, v in zip(prix, x_numpy)])

Les écarts sont minuscules, mais pas forcément nuls : NumPy calcule avec environ 16 chiffres significatifs. Le notebook 2.4 montre quand cette petite imprécision devient un vrai problème.

### Quand le premier pivot est nul

Au sommet de la première colonne, il faut un coefficient non nul pour éliminer en dessous. Que se passe-t-il s'il vaut 0 ?

In [ ]:
A0 = [[0, 1, 1], [1, 1, 1], [2, 3, 1]]
b0 = [5, 6, 13]
M0 = pivot_de_gauss(A0, b0)
print("Solution :", [str(v) for v in remontee(M0)])

La fonction échange les lignes : c'est la première opération autorisée. Un humain le fait sans y penser ; un programme doit le prévoir explicitement.

### Combien de temps pour un grand système ?

Pour $n$ inconnues, le pivot de Gauss fait environ $n^3/3$ multiplications. Comparons notre fonction en Python « pur » avec NumPy, qui appelle une bibliothèque compilée et optimisée.

In [ ]:
import time
import matplotlib.pyplot as plt

def gauss_flottants(A, b):
    # Même algorithme que pivot_de_gauss, en nombres à virgule et sans affichage
    M = [list(map(float, ligne)) + [float(bi)] for ligne, bi in zip(A, b)]
    n = len(M)
    for col in range(n):
        p = max(range(col, n), key=lambda i: abs(M[i][col]))
        M[col], M[p] = M[p], M[col]
        for i in range(col + 1, n):
            f = M[i][col] / M[col][col]
            M[i] = [x - f * y for x, y in zip(M[i], M[col])]
    x = [0.0] * n
    for i in reversed(range(n)):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))) / M[i][i]
    return x

rng = np.random.default_rng(0)
tailles = [25, 50, 100, 200]
t_python, t_numpy = [], []
for n in tailles:
    An, bn = rng.random((n, n)) + n * np.eye(n), rng.random(n)
    debut = time.perf_counter(); gauss_flottants(An.tolist(), bn.tolist()); t_python.append(time.perf_counter() - debut)
    debut = time.perf_counter(); np.linalg.solve(An, bn); t_numpy.append(time.perf_counter() - debut)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(tailles, t_python, "o-", label="pivot en Python pur")
ax.loglog(tailles, t_numpy, "s-", label="np.linalg.solve")
ax.set_xticks(tailles, [str(n) for n in tailles]); ax.minorticks_off()
ax.set_xlabel("nombre d'inconnues n"); ax.set_ylabel("temps (s)")
ax.set_title("Résoudre un système de n équations"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()
for n, tp, tn in zip(tailles, t_python, t_numpy):
    print(f"n = {n:4d} : Python {tp:8.4f} s   NumPy {tn:8.5f} s   rapport {tp / tn:7.0f}")

## ⑤ Interpréter

In [ ]:
print(f"Votre pari pour une rose : {pari_prix_rose} € ; prix calculé : {float(prix[0]):.2f} €")

**Activité** :

1. Rédigez la fiche de prix de la fleuriste.
2. Sur le graphique, les points du pivot en Python sont presque alignés, avec une pente proche de 3. Que signifie une pente de 3 sur un graphique à échelles logarithmiques ? Relisez la phrase sur les $n^3/3$ multiplications.
3. Les deux courbes ont-elles la même pente ? Si oui, d'où vient l'écart entre elles ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Comparer les outils.** Complétez le tableau.

| Outil | Résultat exact ? | Montre les étapes ? | Rapide pour $n = 1\,000$ ? | Quand le choisir ? |
|---|---|---|---|---|
| pivot à la main | … | … | … | … |
| `pivot_de_gauss` (fractions) | … | … | … | … |
| SymPy `rref` | … | … | … | … |
| `np.linalg.solve` | … | … | … | … |

- **Vérifier autrement.** Une quatrième commande arrive : 2 roses, 2 tulipes et 2 lys pour 12 €. Est-ce cohérent avec les prix trouvés ? Que conclure si ce n'était pas le cas ?
- **Généraliser.** Que fait `pivot_de_gauss` sur un système de 3 équations à 2 inconnues ? Sur 2 équations à 3 inconnues ? Essayez, puis gardez vos observations pour le notebook 2.3.

In [ ]:
# Essais pour la dernière question

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
r, t, l = prix
print("Commande 2 roses + 2 tulipes + 2 lys :", 2 * r + 2 * t + 2 * l, "€ — cohérent :", 2 * r + 2 * t + 2 * l == 12)

print("\n3 équations, 2 inconnues (r + t = 3, r - t = 1, 2r + t = 5) :")
pivot_de_gauss([[1, 1], [1, -1], [2, 1]], [3, 1, 5])
print("La dernière ligne devient 0 = 0 : la troisième équation était une conséquence des deux autres.")

print("\n2 équations, 3 inconnues :")
pivot_de_gauss([[1, 1, 1], [1, 2, 3]], [6, 14])
print("Il reste plus d'inconnues que de pivots : une inconnue reste libre, d'où une infinité de solutions.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Avoir l'intuition du comportement de l'outil": 1,
    "Connaître des outils comparables": 1,
    "Comparer les mérites des outils dans le contexte": 1,
    "Corroborer ses résultats": 1,
    "Optimiser la rapidité et la réutilisabilité du calcul": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "2.2-pivot-de-gauss", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")